# PEFT 进阶操作

## 1. 自定义模型适配

In [1]:
import torch
from torch import nn
from peft import LoraConfig, get_peft_model, PeftModel

In [2]:
net1 = nn.Sequential(
    nn.Linear(10, 10),
    nn.ReLU(),
    nn.Linear(10, 2)
)
net1

Sequential(
  (0): Linear(in_features=10, out_features=10, bias=True)
  (1): ReLU()
  (2): Linear(in_features=10, out_features=2, bias=True)
)

In [3]:
for name, param in net1.named_parameters():
    print(name)

0.weight
0.bias
2.weight
2.bias


In [4]:
config = LoraConfig(target_modules=["0"])

In [5]:
model1 = get_peft_model(net1, config)

In [6]:
model1

PeftModel(
  (base_model): LoraModel(
    (model): Sequential(
      (0): lora.Linear(
        (base_layer): Linear(in_features=10, out_features=10, bias=True)
        (lora_dropout): ModuleDict(
          (default): Identity()
        )
        (lora_A): ModuleDict(
          (default): Linear(in_features=10, out_features=8, bias=False)
        )
        (lora_B): ModuleDict(
          (default): Linear(in_features=8, out_features=10, bias=False)
        )
        (lora_embedding_A): ParameterDict()
        (lora_embedding_B): ParameterDict()
      )
      (1): ReLU()
      (2): Linear(in_features=10, out_features=2, bias=True)
    )
  )
)

## 2. 多适配器加载与切换

In [7]:
net2 = nn.Sequential(
    nn.Linear(10, 10),
    nn.ReLU(),
    nn.Linear(10, 2)
)
net2

Sequential(
  (0): Linear(in_features=10, out_features=10, bias=True)
  (1): ReLU()
  (2): Linear(in_features=10, out_features=2, bias=True)
)

In [ ]:
config1 = LoraConfig(target_modules=["0"])
model2 = get_peft_model(net2, config1)
model2.save_pretrained("./loraA")

In [ ]:
config2 = LoraConfig(target_modules=["2"])
model2 = get_peft_model(net2, config2)
model2.save_pretrained("./loraB")

In [2]:
net2 = nn.Sequential(
    nn.Linear(10, 10),
    nn.ReLU(),
    nn.Linear(10, 2)
)
net2

Sequential(
  (0): Linear(in_features=10, out_features=10, bias=True)
  (1): ReLU()
  (2): Linear(in_features=10, out_features=2, bias=True)
)

In [3]:
model2 = PeftModel.from_pretrained(net2, model_id="./loraA/", adapter_name="loraA")
model2

PeftModel(
  (base_model): LoraModel(
    (model): Sequential(
      (0): lora.Linear(
        (base_layer): Linear(in_features=10, out_features=10, bias=True)
        (lora_dropout): ModuleDict(
          (loraA): Identity()
        )
        (lora_A): ModuleDict(
          (loraA): Linear(in_features=10, out_features=8, bias=False)
        )
        (lora_B): ModuleDict(
          (loraA): Linear(in_features=8, out_features=10, bias=False)
        )
        (lora_embedding_A): ParameterDict()
        (lora_embedding_B): ParameterDict()
      )
      (1): ReLU()
      (2): Linear(in_features=10, out_features=2, bias=True)
    )
  )
)

In [4]:
model2.load_adapter("./loraB/", adapter_name="loraB")
model2

PeftModel(
  (base_model): LoraModel(
    (model): Sequential(
      (0): lora.Linear(
        (base_layer): Linear(in_features=10, out_features=10, bias=True)
        (lora_dropout): ModuleDict(
          (loraA): Identity()
        )
        (lora_A): ModuleDict(
          (loraA): Linear(in_features=10, out_features=8, bias=False)
        )
        (lora_B): ModuleDict(
          (loraA): Linear(in_features=8, out_features=10, bias=False)
        )
        (lora_embedding_A): ParameterDict()
        (lora_embedding_B): ParameterDict()
      )
      (1): ReLU()
      (2): lora.Linear(
        (base_layer): Linear(in_features=10, out_features=2, bias=True)
        (lora_dropout): ModuleDict(
          (loraB): Identity()
        )
        (lora_A): ModuleDict(
          (loraB): Linear(in_features=10, out_features=8, bias=False)
        )
        (lora_B): ModuleDict(
          (loraB): Linear(in_features=8, out_features=2, bias=False)
        )
        (lora_embedding_A): ParameterDi

In [6]:
model2.active_adapter

'loraA'

In [10]:
model2(torch.arange(0, 10).view(1, 10).float())

tensor([[-0.3175, -0.8508]], grad_fn=<AddmmBackward0>)

In [5]:
for name, param in model2.named_parameters():
    print(name, param)

base_model.model.0.base_layer.weight Parameter containing:
tensor([[ 0.2142,  0.1430, -0.0410,  0.2093,  0.1331,  0.1879,  0.0777, -0.1375,
         -0.1172,  0.0866],
        [ 0.2686, -0.2997,  0.2070, -0.2941, -0.2661, -0.0133, -0.1796, -0.1869,
          0.2977,  0.0810],
        [-0.2470, -0.0255,  0.2470,  0.1081, -0.0937, -0.0648,  0.2469, -0.0245,
          0.2989, -0.1542],
        [ 0.1685,  0.2920, -0.0128, -0.1566,  0.0004, -0.0106,  0.1239, -0.1090,
          0.2188, -0.0692],
        [-0.0122,  0.0785, -0.0345, -0.0115, -0.1627, -0.2057,  0.0779, -0.2332,
         -0.0573,  0.2458],
        [ 0.2714, -0.0345, -0.3003, -0.1011,  0.0428, -0.0865,  0.3023, -0.3035,
          0.1327,  0.0916],
        [ 0.3140, -0.0065,  0.1719, -0.0601, -0.1981,  0.1434, -0.1915, -0.0074,
         -0.0433, -0.2910],
        [ 0.1288,  0.1271,  0.0367,  0.2254,  0.1079, -0.2632, -0.3030, -0.1235,
         -0.2058,  0.2679],
        [-0.0789,  0.1364,  0.0367,  0.2788, -0.2703, -0.1349, -0.140

In [12]:
for name, param in model2.named_parameters():
    if name in ["base_model.model.0.lora_A.loraA.weight", "base_model.model.0.lora_B.loraA.weight"]:
        param.data = torch.ones_like(param)

In [13]:
model2(torch.arange(0, 10).view(1, 10).float())

tensor([[ -53.9497, -469.4361]], grad_fn=<AddmmBackward0>)

In [15]:
model2.set_adapter("loraB")

In [16]:
model2.active_adapter

'loraB'

In [17]:
model2(torch.arange(0, 10).view(1, 10).float())

tensor([[-0.3175, -0.8508]], grad_fn=<AddBackward0>)

## 3. 禁用适配器

In [18]:
model2.set_adapter("loraA")

In [19]:
model2(torch.arange(0, 10).view(1, 10).float())

tensor([[ -53.9497, -469.4361]], grad_fn=<AddmmBackward0>)

In [20]:
with model2.disable_adapter():
    print(model2(torch.arange(0, 10).view(1, 10).float()))

tensor([[-0.3175, -0.8508]])
